In [30]:
import os
import time
import requests
import json
import re
from urllib.parse import urljoin, quote

# ===== НАСТРОЙКИ =====
BASE_URL = "https://sudact.ru"
AJAX_URL = "https://sudact.ru/magistrate/doc_ajax/"
JUDGE_NAME = "ФИО Судьи"
OUTPUT_DIR = "/Users/haraka/PycharmProjects/study/social/001_1_3_Определение_объёма_эмпирического_материала/data_assembler/sudact_cases"

# Куки из вашего браузера
COOKIES = {
    '_ym_uid': '1765036528422670708',
    '_ym_d': '1765036528',
    '_ym_isad': '2',
    '_ym_visorc': 'b',
}

# Точные заголовки из браузера
HEADERS = {
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Connection': 'keep-alive',
    'Referer': f'https://sudact.ru/magistrate/doc/?magistrate-judge={quote(JUDGE_NAME)}',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
    'X-Requested-With': 'XMLHttpRequest'
}

DELAY = 1

def get_ajax_url(page):
    """Формирует URL для AJAX-запроса"""
    params = {
        'magistrate-judge': JUDGE_NAME,
        '_': int(time.time() * 1000)
    }
    if page > 1:
        params['page'] = page
    
    query_parts = []
    for key, value in params.items():
        if key == 'magistrate-judge':
            encoded_value = quote(value)
        else:
            encoded_value = str(value)
        query_parts.append(f"{key}={encoded_value}")
    
    return f"{AJAX_URL}?{'&'.join(query_parts)}"

def fetch_json(session, page):
    """Получает JSON с данными о делах"""
    url = get_ajax_url(page)
    print(f"Запрос страницы {page}...")
    
    try:
        response = session.get(url, timeout=15)
        response.encoding = 'utf-8'
        response.raise_for_status()
        
        # Сохраняем JSON для отладки
        with open(f'debug_page_{page}.json', 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        # Проверяем, что пришло в ответе
        data = response.json()
        if 'content' in data and data['content']:
            print(f"  ✓ Поле 'content' не пустое, длина: {len(data['content'])}")
        else:
            print(f"  ⚠ Поле 'content' пустое или отсутствует")
        
        return data
    except Exception as e:
        print(f"Ошибка: {e}")
        return None

def parse_json_response(data, page):
    """Извлекает ссылки на дела из JSON-ответа - ИСПРАВЛЕННАЯ ВЕРСИЯ"""
    links = []
    
    if not data or 'content' not in data:
        print(f"  Страница {page}: нет поля 'content' в JSON")
        return links
    
    html_content = data['content']
    
    # Сохраняем HTML для отладки
    with open(f'debug_page_{page}.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    # ШАГ 1: Находим контейнер с результатами
    results_pattern = r'<ul class="results">(.*?)</ul>'
    results_match = re.search(results_pattern, html_content, re.DOTALL)
    
    if not results_match:
        print(f"  Страница {page}: не найден контейнер <ul class='results'>")
        return links
    
    results_html = results_match.group(1)
    
    # ШАГ 2: Внутри контейнера ищем все ссылки на дела
    # Паттерн для ссылок: <a href="/magistrate/doc/...">...</a>
    link_pattern = r'<a\s+href="(/magistrate/doc/[^"]+)"[^>]*>([^<]+)</a>'
    matches = re.findall(link_pattern, results_html)
    
    for href, title in matches:
        # Любая ссылка внутри results - это ссылка на дело
        # Очищаем URL от параметров (берем только часть до ?)
        base_href = href.split('?')[0]
        full_url = urljoin(BASE_URL, base_href)
        clean_title = re.sub(r'\s+', ' ', title).strip()
        links.append((full_url, clean_title))
        print(f"    Найдена ссылка: {clean_title[:50]}...")
    
    # Убираем дубликаты на всякий случай
    unique = []
    seen = set()
    for url, title in links:
        if url not in seen:
            seen.add(url)
            unique.append((url, title))
    
    print(f"  Страница {page}: найдено {len(unique)} дел")
    return unique

def get_total_pages(first_data):
    """Определяет общее количество страниц из первого JSON - ИСПРАВЛЕННАЯ ВЕРСИЯ"""
    if not first_data:
        print("⚠ Нет данных для определения количества страниц")
        return 1
    
    # СПОСОБ 1: Используем поле 'total_found' с количеством документов
    total_found_html = first_data.get('total_found', '')
    
    # В вашем JSON это поле выглядит так: "total_found": "\n\n\nНайдено 1 882 документа"
    match = re.search(r'Найдено\s+([\d\s]+)\s+документов?', total_found_html)
    if match:
        total_docs_str = match.group(1).replace(' ', '')
        try:
            total_docs = int(total_docs_str)
            # Округляем вверх до целого числа страниц (по 10 документов на страницу)
            pages = (total_docs + 9) // 10
            print(f"📊 Найдено {total_docs} документов → {pages} страниц")
            return pages
        except ValueError:
            print(f"⚠ Не удалось преобразовать '{total_docs_str}' в число")
    
    # СПОСОБ 2: Если не нашли, ищем последнюю страницу через пагинацию
    html_content = first_data.get('content', '')
    
    # Ищем все ссылки на страницы
    page_links = re.findall(r'<a href="\?page=(\d+)[^>]*>(\d+)</a>', html_content)
    if page_links:
        pages = [int(link[1]) for link in page_links]
        max_page = max(pages)
        print(f"📊 Найдены страницы {sorted(pages)} в пагинации")
        
        # Если есть многоточие, значит страниц больше, чем показано
        if '&hellip;' in html_content:
            # Обычно показывают около 8 страниц, но реально их больше
            # Возвращаем максимум из найденного + запас
            print(f"⚠ Найдено многоточие, реально страниц больше {max_page}")
            # Временное решение - вернем 189 (вы можете заменить на точное число)
            return 28
    
    print("⚠ Не удалось определить количество страниц, возвращаю 189 (значение по умолчанию)")
    return 28  # Временное решение

def parse_case_page(case_url, session):
    """Парсит страницу конкретного дела"""
    try:
        response = session.get(case_url, headers={
            'User-Agent': HEADERS['User-Agent'],
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
        }, timeout=10)
        response.encoding = 'utf-8'
        html = response.text
    except Exception as e:
        print(f"Ошибка загрузки: {e}")
        return None
    
    # Заголовок
    title_match = re.search(r'<h1>(.*?)</h1>', html, re.DOTALL)
    title = title_match.group(1) if title_match else ""
    
    # Текст дела
    content_match = re.search(r'<td class="h-col1[^>]*>(.*?)</td>', html, re.DOTALL)
    if content_match:
        content = content_match.group(1)
        content = re.sub(r'<[^>]+>', ' ', content)
        content = re.sub(r'<script.*?</script>', '', content, flags=re.DOTALL)
        content = re.sub(r'\s+', ' ', content).strip()
    else:
        content = "Не удалось извлечь текст дела"
    
    return f"{title}\n{'='*60}\nИсточник: {case_url}\n{'='*60}\n\n{content}"

def create_safe_filename(url, title):
    """Создает безопасное имя файла"""
    doc_id = re.search(r'/doc/([^/?]+)', url)
    doc_id = doc_id.group(1) if doc_id else "unknown"
    
    case_num_match = re.search(r'№\s*([^-\s]+(?:-\d+)?)', title)
    case_num = case_num_match.group(1) if case_num_match else ""
    
    if case_num:
        return f"{doc_id}_{case_num}.txt"
    else:
        safe_title = re.sub(r'[\\/*?:"<>|]', '', title)[:50]
        return f"{doc_id}_{safe_title}.txt"

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    session = requests.Session()
    session.headers.update(HEADERS)
    session.cookies.update(COOKIES)
    
    print(f"🔍 Поиск дел судьи: {JUDGE_NAME}")
    print("="*50)
    
    # Загружаем первую страницу
    print("Загрузка первой страницы...")
    first_data = fetch_json(session, 1)
    
    if not first_data:
        print("❌ Не удалось загрузить данные")
        return
    
    # Определяем количество страниц
    total_pages = get_total_pages(first_data)
    
    # Сбор ссылок
    all_links = []
    print("\n📝 Сбор ссылок...")
    
    for page in range(1, total_pages + 1):
        if page == 1:
            data = first_data
        else:
            data = fetch_json(session, page)
        
        if data:
            links = parse_json_response(data, page)
            all_links.extend(links)
        else:
            print(f"Страница {page}: ошибка загрузки")
        
        time.sleep(DELAY)
    
    print(f"\n✅ Всего собрано ссылок: {len(all_links)}")
    
    # Скачивание дел
    print("\n📥 Скачивание дел...")
    success = 0
    failed = 0
    
    for i, (url, title) in enumerate(all_links, 1):
        filename = create_safe_filename(url, title)
        filepath = os.path.join(OUTPUT_DIR, filename)
        
        if os.path.exists(filepath):
            print(f"[{i}/{len(all_links)}] ⏭ Пропуск: {filename}")
            continue
        
        print(f"[{i}/{len(all_links)}] Загрузка: {title[:50]}...")
        
        text = parse_case_page(url, session)
        if text:
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(text)
            success += 1
            print(f"  ✓ Сохранено как {filename}")
        else:
            failed += 1
            print(f"  ✗ Ошибка")
        
        time.sleep(DELAY)
    
    print(f"\n{'='*50}")
    print(f"✅ Готово!")
    print(f"   Успешно: {success}")
    print(f"   Ошибок: {failed}")
    print(f"   Файлы в папке: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

🔍 Поиск дел судьи: Соломатина Галина Михайловна
Загрузка первой страницы...
Запрос страницы 1...
  ✓ Поле 'content' не пустое, длина: 9339
📊 Найдены страницы [2, 3, 4, 5, 6, 7, 8] в пагинации
⚠ Найдено многоточие, реально страниц больше 8

📝 Сбор ссылок...
    Найдена ссылка: Постановление от 26 августа 2025 г. по делу № 5-56...
    Найдена ссылка: Постановление от 26 августа 2025 г. по делу № 5-57...
    Найдена ссылка: Постановление от 26 августа 2025 г. по делу № 5-56...
    Найдена ссылка: Постановление от 26 августа 2025 г. по делу № 5-55...
    Найдена ссылка: Постановление от 26 августа 2025 г. по делу № 5-55...
    Найдена ссылка: Постановление от 26 августа 2025 г. по делу № 5-55...
    Найдена ссылка: Постановление от 26 августа 2025 г. по делу № 5-56...
    Найдена ссылка: Постановление от 30 июля 2025 г. по делу № 5-563/2...
    Найдена ссылка: Постановление от 28 мая 2025 г. по делу № 5-385/20...
    Найдена ссылка: Постановление от 28 мая 2025 г. по делу № 5-390/20...
  С